# Section 4: Ransomware detection

Compare a linear SVM with an LSTM using ordered Windows API-call sequences. Keep only goodware and ransomware from the multiclass source dataset.

This revision excludes every sequence with conflicting labels, adds sequence exploration, uses raw SVM margins, selects validation thresholds under a false-alert budget, records validation loss and provenance, and exports manifests with artifacts. Outputs were cleared; rerun top to bottom.

## Setup

The first cell changes folders only in Colab. For a local kernel, start from the repository root. Package versions are not pinned.

In [ ]:
import os
import sys

# Use Colab's working folder without changing local notebook paths.
if "google.colab" in sys.modules:
    os.chdir("/content")

In [ ]:
%pip install -q joblib matplotlib numpy pandas scikit-learn torch

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import random
import shutil
import urllib.request
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from torch import nn
from torch.utils.data import DataLoader, Dataset

### Experiment settings

Keep seed, epoch count, threshold policy and output folders together. Fifteen epochs define this revision; no older saved metric should be attached to it. The run summary records dataset hash, packages, device and split membership.

In [ ]:
seed = 42
epochs = 15
false_alert_budget = 0.01
run_id = datetime.now(timezone.utc).strftime("s04-%Y%m%dT%H%M%SZ")

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "notebooks").is_dir() and (candidate / "docs").is_dir():
            return candidate
    return Path.cwd()

in_colab = "google.colab" in sys.modules
root = Path("/content/section_04_workspace") if in_colab else find_project_root()
data_file = root / "data/raw/ransomware-api/multiclass_malware_api_seq.csv"
processed_dir = root / "data/processed/section_04"
model_dir = root / "models/section_04"
results_dir = root / "reports/section_04"
for directory in (data_file.parent, processed_dir, model_dir, results_dir / "metrics", results_dir / "figures"):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Load the API sequences

Download the labelled sequence CSV if it is not already available. This is recorded API-call data, not executable malware.

In [ ]:
data_url = "https://zenodo.org/records/16742661/files/multiclass_malware_api_seq.csv?download=1"
if not data_file.exists():
    urllib.request.urlretrieve(data_url, data_file)
dataset_hash = hashlib.sha256(data_file.read_bytes()).hexdigest()

In [ ]:
raw = pd.read_csv(data_file)
selected = raw.loc[raw["TYPE"].isin(["Goodware", "ransomware"]), ["APISEQ", "TYPE"]].dropna().copy()
label_counts = selected.groupby("APISEQ")["TYPE"].nunique()
conflicting_sequences = set(label_counts[label_counts > 1].index)
conflicts = selected[selected["APISEQ"].isin(conflicting_sequences)].copy()
conflicts["sequence_sha256"] = conflicts["APISEQ"].map(
    lambda value: hashlib.sha256(value.encode()).hexdigest()
)
conflicts.groupby(["sequence_sha256", "TYPE"]).size().rename("rows").reset_index().to_csv(
    processed_dir / "conflicting-sequences.csv", index=False
)

# Exclude every row belonging to an ambiguous sequence, then deduplicate unambiguous traces.
data = selected.loc[~selected["APISEQ"].isin(conflicting_sequences)].drop_duplicates("APISEQ").reset_index(drop=True)
data.insert(0, "sequence_id", range(len(data)))
data["sequence_sha256"] = data["APISEQ"].map(lambda value: hashlib.sha256(value.encode()).hexdigest())
data["label"] = data["TYPE"].eq("ransomware").astype(int)
data["sequence_length"] = data["APISEQ"].str.split().str.len()
profile = {
    "source_records": len(raw), "selected_nonmissing_records": len(selected),
    "conflicting_unique_sequences_excluded": len(conflicting_sequences),
    "conflicting_rows_excluded": len(conflicts),
    "records_after_conflict_exclusion_and_deduplication": len(data),
    "goodware": int((data["label"] == 0).sum()),
    "ransomware": int((data["label"] == 1).sum()),
    "unique_api_calls": len(set(" ".join(data["APISEQ"]).split())),
}
(processed_dir / "data-profile.json").write_text(json.dumps(profile, indent=2))
display(pd.Series(profile).to_frame("value"))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
data["TYPE"].value_counts().reindex(["Goodware", "ransomware"]).plot.bar(
    color=["#4C78A8", "#E45756"], rot=0, ylabel="Unique sequences",
    title="Class distribution", ax=axes[0],
)
data.boxplot(column="sequence_length", by="TYPE", showfliers=False, ax=axes[1])
axes[1].set(title="Sequence length", xlabel="Class", ylabel="API calls")
fig.suptitle("")
fig.tight_layout()
fig.savefig(results_dir / "figures/data-exploration.png", dpi=180)
plt.show()

All rows belonging to a sequence with more than one label are excluded before deduplication. This removes row-order dependence and leaves 590 unique goodware sequences and 38 unique ransomware sequences in the audited dataset. The small positive class remains a major statistical limitation.

## Split the data

Use the same stratified 70/15/15 split for both models and save the sequence membership. In the recorded split, validation and test each contain only six ransomware traces.

In [ ]:
train_val, test = train_test_split(
    data, test_size=0.15, stratify=data["label"], random_state=seed,
)
train, val = train_test_split(
    train_val, test_size=0.15 / 0.85,
    stratify=train_val["label"], random_state=seed,
)
train, val, test = [frame.reset_index(drop=True) for frame in (train, val, test)]

manifest = pd.concat([
    frame[["sequence_id", "sequence_sha256", "TYPE", "label"]].assign(split=name)
    for name, frame in (("train", train), ("validation", val), ("test", test))
], ignore_index=True)
manifest.to_csv(processed_dir / "split-manifest.csv", index=False)
display(pd.crosstab(manifest["split"], manifest["TYPE"]))

## Evaluate the models

Choose each cutoff on validation data under a 1% false-positive-rate budget and lock it before test evaluation. The SVM uses its raw decision margin; it is a ranking score, not a probability.

In [ ]:
def select_threshold(labels, scores, max_fpr=false_alert_budget):
    labels, scores = np.asarray(labels), np.asarray(scores)
    thresholds = np.r_[np.inf, np.unique(scores)[::-1]]
    candidates = []
    for threshold in thresholds:
        preds = scores >= threshold
        tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
        fpr = fp / max(fp + tn, 1)
        if fpr <= max_fpr:
            candidates.append((f1_score(labels, preds, zero_division=0), -fpr, threshold))
    return max(candidates)[2]

def show_results(name, labels, scores, threshold, filename):
    preds = (scores >= threshold).astype(int)
    metrics = {
        "run_id": run_id, "threshold": float(threshold),
        "validation_false_alert_budget": false_alert_budget,
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, zero_division=0)),
        "recall": float(recall_score(labels, preds, zero_division=0)),
        "f1": float(f1_score(labels, preds, zero_division=0)),
        "roc_auc": float(roc_auc_score(labels, scores)),
        "average_precision": float(average_precision_score(labels, scores)),
        "confusion_matrix": confusion_matrix(labels, preds).tolist(),
        "classification_report": classification_report(
            labels, preds, target_names=["goodware", "ransomware"], output_dict=True, zero_division=0,
        ),
    }
    (results_dir / "metrics" / f"{filename}.json").write_text(json.dumps(metrics, indent=2))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ConfusionMatrixDisplay.from_predictions(
        labels, preds, display_labels=["Goodware", "Ransomware"],
        cmap="Blues", colorbar=False, ax=axes[0],
    )
    fpr, tpr, _ = roc_curve(labels, scores)
    axes[1].plot(fpr, tpr)
    axes[1].plot([0, 1], [0, 1], "--", color="grey")
    axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title=f"ROC AUC: {metrics['roc_auc']:.3f}")
    precision, recall, _ = precision_recall_curve(labels, scores)
    axes[2].plot(recall, precision)
    axes[2].set(xlabel="Recall", ylabel="Precision", title=f"Average precision: {metrics['average_precision']:.3f}")
    fig.suptitle(name)
    fig.tight_layout()
    fig.savefig(results_dir / "figures" / f"{filename}-evaluation.png", dpi=180)
    plt.show()
    return metrics

## Linear SVM

Represent individual API calls and adjacent pairs with TF-IDF, preserving their spelling and case. Fit the vocabulary and class-balanced linear SVM on training sequences only. These features capture short transitions, not the complete call order.

In [ ]:
svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=False, token_pattern=r"\S+", ngram_range=(1, 2),
        max_features=5000, sublinear_tf=True,
    )),
    ("classifier", SVC(kernel="linear", class_weight="balanced", random_state=seed)),
])
svm_model.fit(train["APISEQ"], train["label"])
svm_val_scores = svm_model.decision_function(val["APISEQ"])
svm_threshold = select_threshold(val["label"], svm_val_scores)
svm_scores = svm_model.decision_function(test["APISEQ"])
svm_metrics = show_results(
    "TF-IDF API n-grams with linear SVM", test["label"].to_numpy(),
    svm_scores, svm_threshold, "svm",
)
joblib.dump({"model": svm_model, "threshold": svm_threshold, "run_id": run_id}, model_dir / "svm.joblib")
display(pd.DataFrame(svm_metrics["classification_report"]).T)

## LSTM

Build the API vocabulary from training sequences, reserving IDs for padding and unknown calls. Keep up to 100 calls in their original order; all selected source traces currently have that length.

In [ ]:
max_len = 100
batch_size = 32
# Build the vocabulary from training API calls only.
api_counts = Counter(api for sequence in train["APISEQ"] for api in sequence.split())
vocab = {"<PAD>": 0, "<UNK>": 1}
vocab.update({api: index for index, (api, _) in enumerate(api_counts.most_common(), 2)})

def encode_calls(sequence):
    # Replace unseen calls with UNK and keep the first 100 calls.
    tokens = [vocab.get(api, 1) for api in sequence.split()[:max_len]]
    length = len(tokens)
    # Pad shorter traces with zeros so every batch has the same width.
    return tokens + [0] * (max_len - length), length

### Prepare the dataset

Store each trace's token IDs, true length and binary label as tensors for batching.

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, frame):
        # Convert each trace to tensors once, ready for the data loader.
        encoded = [encode_calls(sequence) for sequence in frame["APISEQ"]]
        self.tokens = torch.tensor([item[0] for item in encoded], dtype=torch.long)
        self.lengths = torch.tensor([item[1] for item in encoded], dtype=torch.long)
        self.labels = torch.tensor(frame["label"].to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.tokens[index], self.lengths[index], self.labels[index]

### Define the model

Learn 64-dimensional API embeddings and feed them through a 64-unit LSTM. Dropout regularizes the final classifier, which returns one logit per trace.

In [ ]:
class ApiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 64, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.output = nn.Linear(64, 1)

    def forward(self, tokens, lengths):
        # Turn API IDs into learned vectors and read them in order with the LSTM.
        sequence, _ = self.lstm(self.embedding(tokens))
        rows = torch.arange(len(lengths), device=tokens.device)
        # Use the last real call, not a padding position.
        return self.output(self.dropout(sequence[rows, lengths - 1])).squeeze(1)

def predict_lstm(model, loader, device):
    # Turn off dropout and gradient tracking while making predictions.
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for tokens, lengths, batch_labels in loader:
            logits = model(tokens.to(device), lengths.to(device))
            # Sigmoid turns each logit into a ransomware score between 0 and 1.
            probs.extend(torch.sigmoid(logits).cpu().numpy())
            labels.extend(batch_labels.numpy())
    return np.asarray(labels, dtype=int), np.asarray(probs)

### Set up training

Shuffle training batches, use CUDA if available and weight the ransomware class by the training class ratio. Binary cross-entropy takes logits directly; sigmoid is used when making prediction scores.

In [ ]:
# Shuffle training batches while keeping validation and test order fixed.
generator = torch.Generator().manual_seed(seed)
train_loader = DataLoader(SequenceDataset(train), batch_size=batch_size, shuffle=True, generator=generator)
val_loader = DataLoader(SequenceDataset(val), batch_size=batch_size)
test_loader = DataLoader(SequenceDataset(test), batch_size=batch_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lstm_model = ApiLSTM().to(device)
# Give ransomware examples more weight because they are much rarer.
pos_weight = torch.tensor([(len(train) - train["label"].sum()) / train["label"].sum()], device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=0.001)

### Train and evaluate

Run all 15 epochs, record training loss, validation loss and validation F1, then restore the lowest-validation-loss checkpoint. Save the numerical history and chosen epoch. Threshold selection remains noisy because validation contains only a handful of ransomware traces.

In [ ]:
def validation_loss(model, loader):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for tokens, lengths, labels in loader:
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)
            total += loss_fn(model(tokens, lengths), labels).item() * len(labels)
    return total / len(loader.dataset)

history = []
best_val_loss = float("inf")
for epoch in range(1, epochs + 1):
    lstm_model.train()
    total_loss = 0
    for tokens, lengths, labels in train_loader:
        tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(lstm_model(tokens, lengths), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    y_val, val_scores = predict_lstm(lstm_model, val_loader, device)
    val_loss = validation_loss(lstm_model, val_loader)
    val_f1 = f1_score(y_val, val_scores >= 0.5, zero_division=0)
    history.append({"epoch": epoch, "training_loss": total_loss / len(train),
                    "validation_loss": val_loss, "validation_f1_at_0_5": val_f1})
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        chosen_epoch = epoch
        best_state = {name: value.detach().cpu().clone() for name, value in lstm_model.state_dict().items()}

lstm_model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
history_df.to_csv(results_dir / "metrics/lstm-training-history.csv", index=False)
display(history_df)
history_df.set_index("epoch")[["training_loss", "validation_loss"]].plot(figsize=(7, 4), title="LSTM loss history")
plt.tight_layout()
plt.savefig(results_dir / "figures/lstm-training-history.png", dpi=180)
plt.show()

y_val, val_scores = predict_lstm(lstm_model, val_loader, device)
lstm_threshold = select_threshold(y_val, val_scores)
y_test, lstm_scores = predict_lstm(lstm_model, test_loader, device)
lstm_metrics = show_results("API-call LSTM", y_test, lstm_scores, lstm_threshold, "lstm")
lstm_metrics.update({"best_validation_loss": float(best_val_loss), "chosen_epoch": chosen_epoch,
                     "epochs": epochs, "device": str(device)})
(results_dir / "metrics/lstm.json").write_text(json.dumps(lstm_metrics, indent=2))
torch.save({"model_state": best_state, "vocabulary": vocab, "threshold": lstm_threshold,
            "run_id": run_id}, model_dir / "lstm.pt")
display(pd.DataFrame(lstm_metrics["classification_report"]).T)

## Compare the models

Save metrics, histories, conflict audit, split manifest, models and environment details under one run identifier. In Colab, the export includes processed evidence as well as reports and model files.

In [ ]:
metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
comparison = pd.DataFrame([
    {"model": "Linear SVM", **{metric: svm_metrics[metric] for metric in metric_names}},
    {"model": "LSTM", **{metric: lstm_metrics[metric] for metric in metric_names}},
])
comparison.to_csv(results_dir / "model-comparison.csv", index=False)
packages = ["joblib", "matplotlib", "numpy", "pandas", "scikit-learn", "torch"]
(results_dir / "run-summary.json").write_text(json.dumps({
    "run_id": run_id, "created_utc": datetime.now(timezone.utc).isoformat(), "seed": seed,
    "dataset_sha256": dataset_hash,
    "records_after_conflict_exclusion_and_deduplication": len(data),
    "train_records": len(train), "validation_records": len(val), "test_records": len(test),
    "unique_goodware": int((data["label"] == 0).sum()),
    "unique_ransomware": int((data["label"] == 1).sum()),
    "lstm_epochs": epochs, "lstm_chosen_epoch": chosen_epoch,
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": {name: importlib.metadata.version(name) for name in packages},
}, indent=2))
display(comparison.style.format({metric: "{:.4f}" for metric in metric_names}).highlight_max(subset=metric_names, color="#d9ead3"))

if in_colab:
    export_dir = root / "section_04_export"
    shutil.copytree(results_dir, export_dir / "reports", dirs_exist_ok=True)
    shutil.copytree(model_dir, export_dir / "models", dirs_exist_ok=True)
    shutil.copytree(processed_dir, export_dir / "processed", dirs_exist_ok=True)
    shutil.make_archive("/content/section_04_results", "zip", root_dir=export_dir)

## What to take from the results

- The conflict policy is explicit: every API sequence observed with both labels is excluded. The remaining positive class is still extremely small, so report counts and uncertainty with every score.
- Raw SVM margins are ranking scores, not calibrated probabilities. Both thresholds come only from validation data under the stated false-alert budget.
- The dataset has no reliable malware-family grouping field, so random sequence splits do not establish family-independent generalisation. Completed-trace classification is detection, not live blocking or encryption prevention.